<a href="https://colab.research.google.com/github/predicting-pregnancy-success/pregnancy-ml-models/blob/main/%EB%82%9C%EC%9E%84_%ED%99%98%EC%9E%90_%EB%8C%80%EC%83%81_%EC%9E%84%EC%8B%A0_%EC%84%B1%EA%B3%B5_%EC%97%AC%EB%B6%80_%EC%98%88%EC%B8%A1AI_v3_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# rtdl 설치 (FT-Transformer 라이브러리)
!pip install rtdl-revisiting-models -q

# GPU 확인
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
import numpy as np
import pandas as pd
from google.colab import drive

# 마운트 (공백 없는 경로!)
drive.mount('/content/drive')

# 작업 폴더
SAVE_DIR = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# 기존 결과 로드
tuned_oof  = np.load(f'{SAVE_DIR}/tuned_oof.npy')
tuned_test = np.load(f'{SAVE_DIR}/tuned_test.npy')
train_copy = pd.read_parquet(f'{SAVE_DIR}/train_processed.parquet')
test_copy  = pd.read_parquet(f'{SAVE_DIR}/test_processed.parquet')

# 검증
from sklearn.metrics import roc_auc_score
y_check = train_copy['임신 성공 여부'].values
print(f"✓ tuned_oof shape: {tuned_oof.shape}")
print(f"✓ tuned_test shape: {tuned_test.shape}")
print(f"✓ train_copy shape: {train_copy.shape}")
print(f"✓ test_copy shape: {test_copy.shape}")
print(f"✓ CatBoost OOF AUC 재확인: {roc_auc_score(y_check, tuned_oof):.4f}")
print(f"   → 0.7404 정도 나오면 정상!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ tuned_oof shape: (256351,)
✓ tuned_test shape: (90067,)
✓ train_copy shape: (256351, 104)
✓ test_copy shape: (90067, 103)
✓ CatBoost OOF AUC 재확인: 0.7404
   → 0.7404 정도 나오면 정상!


In [5]:
# 타겟/ID 분리
TARGET = '임신 성공 여부'
ID_COL = 'ID'

y = train_copy[TARGET].values
train_features = train_copy.drop(columns=[TARGET, ID_COL])
test_features  = test_copy.drop(columns=[ID_COL])

# 수치형 vs 카테고리형 자동 분리
# 기준: unique 값 20개 이하면 카테고리형 (라벨 인코딩된 컬럼들 잡힘)
cat_cols = []
num_cols = []
for col in train_features.columns:
    nunique = train_features[col].nunique()
    if nunique <= 20:
        cat_cols.append(col)
    else:
        num_cols.append(col)

print(f"수치형 피처 ({len(num_cols)}개)")
print(f"카테고리형 피처 ({len(cat_cols)}개)")
print(f"총합: {len(num_cols) + len(cat_cols)}개")

수치형 피처 (21개)
카테고리형 피처 (81개)
총합: 102개


In [6]:
# 혹시 남아있을 NaN 처리 (안전장치)
for col in num_cols:
    median_val = train_features[col].median()
    train_features[col] = train_features[col].fillna(median_val)
    test_features[col]  = test_features[col].fillna(median_val)

# 카테고리: 음수값 → 0부터 시작하도록 재인코딩 (rtdl 요구사항)
from sklearn.preprocessing import LabelEncoder

cat_cardinalities = []
for col in cat_cols:
    train_features[col] = train_features[col].fillna(-1).astype(int)
    test_features[col]  = test_features[col].fillna(-1).astype(int)

    combined = pd.concat([train_features[col], test_features[col]]).astype(str)
    le = LabelEncoder()
    le.fit(combined)
    train_features[col] = le.transform(train_features[col].astype(str))
    test_features[col]  = le.transform(test_features[col].astype(str))
    cat_cardinalities.append(len(le.classes_))

print(f"✓ 카테고리 cardinality 합: {sum(cat_cardinalities)}")
print(f"✓ 평균 cardinality: {np.mean(cat_cardinalities):.1f}")
print(f"✓ 최대 cardinality: {max(cat_cardinalities)}")

# numpy 변환
X_num_train = train_features[num_cols].values.astype(np.float32)
X_cat_train = train_features[cat_cols].values.astype(np.int64)
X_num_test  = test_features[num_cols].values.astype(np.float32)
X_cat_test  = test_features[cat_cols].values.astype(np.int64)

print(f"\n✓ X_num_train: {X_num_train.shape}")
print(f"✓ X_cat_train: {X_cat_train.shape}")
print(f"✓ X_num_test:  {X_num_test.shape}")
print(f"✓ X_cat_test:  {X_cat_test.shape}")

✓ 카테고리 cardinality 합: 300
✓ 평균 cardinality: 3.7
✓ 최대 cardinality: 14

✓ X_num_train: (256351, 21)
✓ X_cat_train: (256351, 81)
✓ X_num_test:  (90067, 21)
✓ X_cat_test:  (90067, 81)


In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import rtdl_revisiting_models as rtdl

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

class TabularDataset(Dataset):
    def __init__(self, X_num, X_cat, y=None):
        self.X_num = torch.tensor(X_num, dtype=torch.float32)
        self.X_cat = torch.tensor(X_cat, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X_num)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X_num[idx], self.X_cat[idx], self.y[idx]
        return self.X_num[idx], self.X_cat[idx]


def train_ft_fold(X_num_tr, X_cat_tr, y_tr,
                  X_num_val, X_cat_val, y_val,
                  X_num_te, X_cat_te,
                  fold_idx, seed=42,
                  n_epochs=50, batch_size=512, lr=1e-4,
                  weight_decay=1e-5, patience=8):
    torch.manual_seed(seed)
    np.random.seed(seed)

    # 수치형 스케일링 (fold별로 fit)
    scaler = StandardScaler()
    X_num_tr_s  = scaler.fit_transform(X_num_tr).astype(np.float32)
    X_num_val_s = scaler.transform(X_num_val).astype(np.float32)
    X_num_te_s  = scaler.transform(X_num_te).astype(np.float32)

    train_ds = TabularDataset(X_num_tr_s, X_cat_tr, y_tr)
    val_ds   = TabularDataset(X_num_val_s, X_cat_val, y_val)
    test_ds  = TabularDataset(X_num_te_s, X_cat_te)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size*2, shuffle=False)
    test_loader  = DataLoader(test_ds, batch_size=batch_size*2, shuffle=False)

    # ✅ FT-Transformer (최신 rtdl API: n_cont_features)
    model = rtdl.FTTransformer(
        n_cont_features=len(num_cols),       # ← 변경됨
        cat_cardinalities=cat_cardinalities,
        d_out=1,
        n_blocks=3,
        d_block=192,
        attention_n_heads=8,
        attention_dropout=0.2,
        ffn_d_hidden_multiplier=4/3,
        ffn_dropout=0.1,
        residual_dropout=0.0,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    criterion = nn.BCEWithLogitsLoss()

    best_auc = 0.0
    best_val_preds, best_test_preds = None, None
    no_improve = 0

    for epoch in range(n_epochs):
        # Train
        model.train()
        train_losses = []
        for xn, xc, yb in train_loader:
            xn, xc, yb = xn.to(device), xc.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xn, xc).squeeze(-1)   # forward(x_cont, x_cat)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_losses.append(loss.item())
        scheduler.step()

        # Validate
        model.eval()
        val_preds = []
        with torch.no_grad():
            for xn, xc, _ in val_loader:
                xn, xc = xn.to(device), xc.to(device)
                logits = model(xn, xc).squeeze(-1)
                val_preds.append(torch.sigmoid(logits).cpu().numpy())
        val_preds = np.concatenate(val_preds)
        val_auc = roc_auc_score(y_val, val_preds)

        # 개선되면 test 예측 갱신
        if val_auc > best_auc:
            best_auc = val_auc
            best_val_preds = val_preds.copy()
            test_preds = []
            with torch.no_grad():
                for xn, xc in test_loader:
                    xn, xc = xn.to(device), xc.to(device)
                    logits = model(xn, xc).squeeze(-1)
                    test_preds.append(torch.sigmoid(logits).cpu().numpy())
            best_test_preds = np.concatenate(test_preds)
            no_improve = 0
        else:
            no_improve += 1

        if (epoch+1) % 5 == 0 or epoch == 0:
            print(f"  [Fold {fold_idx} | Ep {epoch+1:02d}] "
                  f"loss={np.mean(train_losses):.4f}  val_AUC={val_auc:.4f}  best={best_auc:.4f}")

        if no_improve >= patience:
            print(f"  ⏹ Early stop @ epoch {epoch+1} (best={best_auc:.4f})")
            break

    return best_val_preds, best_test_preds, best_auc

print("✓ 학습 함수 정의 완료")

✓ 학습 함수 정의 완료


In [8]:
from sklearn.model_selection import StratifiedKFold

N_SPLITS = 5
SEED = 42

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_ft = np.zeros(len(y))
test_ft = np.zeros(len(X_num_test))
fold_aucs = []

for fold_idx, (tr_idx, val_idx) in enumerate(skf.split(X_num_train, y), 1):
    print(f"\n========== Fold {fold_idx}/{N_SPLITS} ==========")

    val_preds, test_preds, val_auc = train_ft_fold(
        X_num_train[tr_idx], X_cat_train[tr_idx], y[tr_idx],
        X_num_train[val_idx], X_cat_train[val_idx], y[val_idx],
        X_num_test, X_cat_test,
        fold_idx=fold_idx, seed=SEED,
        n_epochs=50, batch_size=512, lr=1e-4,
        weight_decay=1e-5, patience=8,
    )

    oof_ft[val_idx] = val_preds
    test_ft += test_preds / N_SPLITS
    fold_aucs.append(val_auc)
    print(f"  ✓ Fold {fold_idx} 완료 - best AUC: {val_auc:.4f}")

# 전체 결과
oof_ft_auc = roc_auc_score(y, oof_ft)
print(f"\n{'='*50}")
print(f" FT-Transformer 결과")
print(f"{'='*50}")
print(f"Fold AUCs:        {[f'{a:.4f}' for a in fold_aucs]}")
print(f"Mean Fold AUC:    {np.mean(fold_aucs):.4f} (±{np.std(fold_aucs):.4f})")
print(f"OOF AUC:          {oof_ft_auc:.4f}")
print(f"기존 CatBoost OOF: 0.7404")
print(f"{'='*50}")


========== Fold 1/5 ==========
  [Fold 1 | Ep 01] loss=0.5054  val_AUC=0.7329  best=0.7329
  [Fold 1 | Ep 05] loss=0.4899  val_AUC=0.7358  best=0.7358
  [Fold 1 | Ep 10] loss=0.4884  val_AUC=0.7364  best=0.7364
  [Fold 1 | Ep 15] loss=0.4875  val_AUC=0.7373  best=0.7373
  [Fold 1 | Ep 20] loss=0.4864  val_AUC=0.7373  best=0.7375
  [Fold 1 | Ep 25] loss=0.4855  val_AUC=0.7371  best=0.7375
  ⏹ Early stop @ epoch 26 (best=0.7375)
  ✓ Fold 1 완료 - best AUC: 0.7375

========== Fold 2/5 ==========
  [Fold 2 | Ep 01] loss=0.5055  val_AUC=0.7371  best=0.7371
  [Fold 2 | Ep 05] loss=0.4904  val_AUC=0.7407  best=0.7407
  [Fold 2 | Ep 10] loss=0.4892  val_AUC=0.7411  best=0.7412
  [Fold 2 | Ep 15] loss=0.4884  val_AUC=0.7413  best=0.7419
  [Fold 2 | Ep 20] loss=0.4874  val_AUC=0.7425  best=0.7425
  [Fold 2 | Ep 25] loss=0.4864  val_AUC=0.7421  best=0.7425
  [Fold 2 | Ep 30] loss=0.4855  val_AUC=0.7421  best=0.7425
  ⏹ Early stop @ epoch 32 (best=0.7425)
  ✓ Fold 2 완료 - best AUC: 0.7425

=========

In [10]:
# 가중치 그리드서치
print("\n[가중치별 앙상블 OOF AUC]")
best_w, best_w_auc = 0, 0
for w in np.arange(0.0, 1.01, 0.05):
    ens = w * oof_ft + (1-w) * tuned_oof
    auc = roc_auc_score(y, ens)
    if auc > best_w_auc:
        best_w_auc, best_w = auc, w
    if abs(w*4 - round(w*4)) < 1e-6:
        marker = "" if w == best_w else ""
        print(f"   FT={w:.2f}  Cat={1-w:.2f}  AUC={auc:.4f}{marker}")

print(f"\n 최적: FT={best_w:.2f}, Cat={1-best_w:.2f} → OOF AUC={best_w_auc:.4f}")
print(f"   (CatBoost 단독: 0.7404 → +{(best_w_auc-0.7404)*10000:.1f}bp 향상)")

# 저장
np.save(f'{SAVE_DIR}/oof_ft.npy', oof_ft)
np.save(f'{SAVE_DIR}/test_ft.npy', test_ft)
print(f"\n✓ oof_ft.npy, test_ft.npy 저장 완료")


[가중치별 앙상블 OOF AUC]
   FT=0.00  Cat=1.00  AUC=0.7404
   FT=0.25  Cat=0.75  AUC=0.7408
   FT=0.50  Cat=0.50  AUC=0.7407
   FT=0.75  Cat=0.25  AUC=0.7403
   FT=1.00  Cat=0.00  AUC=0.7395

 최적: FT=0.35, Cat=0.65 → OOF AUC=0.7408
   (CatBoost 단독: 0.7404 → +4.0bp 향상)

✓ oof_ft.npy, test_ft.npy 저장 완료


In [9]:
import numpy as np
import pandas as pd
from google.colab import drive

# 1. Drive 마운트
drive.mount('/content/drive')

# 2. 작업 폴더
SAVE_DIR = '/content/drive/MyDrive/오즈코딩_AI헬스케어/5-2. 2차 해커톤'

# 3. 모든 저장 파일 로드
tuned_oof  = np.load(f'{SAVE_DIR}/tuned_oof.npy')
tuned_test = np.load(f'{SAVE_DIR}/tuned_test.npy')
oof_ft     = np.load(f'{SAVE_DIR}/oof_ft.npy')
test_ft    = np.load(f'{SAVE_DIR}/test_ft.npy')
train_copy = pd.read_parquet(f'{SAVE_DIR}/train_processed.parquet')
test_copy  = pd.read_parquet(f'{SAVE_DIR}/test_processed.parquet')

# 4. 검증 (어제 결과 재현)
from sklearn.metrics import roc_auc_score
y = train_copy['임신 성공 여부'].values

print("=== 어제 결과 재현 확인 ===")
print(f"CatBoost OOF AUC:        {roc_auc_score(y, tuned_oof):.4f}  (예상: 0.7404)")
print(f"FT-Transformer OOF AUC:  {roc_auc_score(y, oof_ft):.4f}    (예상: 0.7395)")
print(f"\n앙상블 (FT 35% + Cat 65%):")
ens = 0.35 * oof_ft + 0.65 * tuned_oof
print(f"  OOF AUC: {roc_auc_score(y, ens):.4f}  (예상: 0.7408)")

print(f"\n=== 데이터 크기 확인 ===")
print(f"train_copy: {train_copy.shape}")
print(f"test_copy:  {test_copy.shape}")
print(f"tuned_oof:  {tuned_oof.shape}")
print(f"oof_ft:     {oof_ft.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
=== 어제 결과 재현 확인 ===
CatBoost OOF AUC:        0.7404  (예상: 0.7404)
FT-Transformer OOF AUC:  0.7395    (예상: 0.7395)

앙상블 (FT 35% + Cat 65%):
  OOF AUC: 0.7408  (예상: 0.7408)

=== 데이터 크기 확인 ===
train_copy: (256351, 104)
test_copy:  (90067, 103)
tuned_oof:  (256351,)
oof_ft:     (256351,)
